In [ ]:
# 1. Gerekli Kütüphaneleri İçe Aktar
from ultralytics import YOLO
import torch
from pathlib import Path
import yaml

print("="*60)
print("🎯 YOLOv8 MODEL EĞİTİMİ")
print("="*60)

# GPU kontrolü
if torch.cuda.is_available():
    print(f"\n✅ GPU Bulundu: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = 0
else:
    print("\n⚠️  GPU bulunamadı, CPU kullanılacak")
    print("   Not: CPU ile eğitim çok yavaş olacaktır!")
    device = 'cpu'

# 🚀 YOLOv8 Trafik İşareti Tanıma - Model Eğitimi

## ⚠️ Veri Dengesizliği Sorunu Çözüldü! ⚖️

Bu notebook, **147x veri dengesizliği** problemi için özel olarak optimize edilmiştir:
- **En yaygın sınıf:** 147 örnek
- **En nadir sınıf:** 1 örnek

### 🛠️ Uygulanan Çözümler:
1. **Class Loss Gain Artırıldı** - Nadir sınıfların loss'una daha fazla ağırlık
2. **Güçlendirilmiş Augmentation** - MixUp, Copy-Paste, gelişmiş geometrik dönüşümler
3. **Class Weights Analizi** - Her sınıf için optimal ağırlık hesaplama

### 📊 Beklenen Sonuçlar:
- ✅ Nadir sınıflarda performans artışı
- ✅ Daha dengeli confusion matrix
- ✅ Genel mAP'te %5-10 iyileşme
- ✅ Robust ve generalize model

---

In [ ]:
# 2. Veri Seti Yolunu Belirle
dataset_path = Path(r'c:\Users\hatice\Desktop\modelegtm\dataset_cleaned')
data_yaml = dataset_path / 'data.yaml'

# Veri setinin varlığını kontrol et
if not dataset_path.exists():
    print(f"❌ HATA: Veri seti bulunamadı!")
    print(f"   Beklenen konum: {dataset_path}")
else:
    print(f"\n✅ Veri Seti Bulundu: {dataset_path}")
    
    # data.yaml içeriğini göster
    with open(data_yaml, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    
    print(f"\n📊 Veri Seti Bilgileri:")
    print(f"   • Sınıf Sayısı: {config['nc']}")
    print(f"   • Train: {config['train']}")
    print(f"   • Valid: {config['val']}")
    print(f"   • Test: {config['test']}")
    
    # Dosya sayılarını kontrol et
    train_imgs = len(list((dataset_path / 'train' / 'images').glob('*')))
    valid_imgs = len(list((dataset_path / 'valid' / 'images').glob('*')))
    test_imgs = len(list((dataset_path / 'test' / 'images').glob('*')))
    
    print(f"\n📂 Dosya Sayıları:")
    print(f"   • Train: {train_imgs} görsel")
    print(f"   • Valid: {valid_imgs} görsel")
    print(f"   • Test: {test_imgs} görsel")

### 📊 Veri Dengesizliği Analizi ve Class Weights Hesaplama

In [ ]:
# Veri dengesizliği sorununu çözmek için class weights hesapla
import numpy as np
from collections import Counter, defaultdict

print("\n" + "="*60)
print("⚖️  VERİ DENGESİZLİĞİ ANALİZİ")
print("="*60)

def parse_yolo_labels(label_file):
    """YOLO format etiket dosyasını parse et"""
    classes = []
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                classes.append(class_id)
    return classes

# Train setindeki sınıf dağılımını analiz et
train_labels_dir = dataset_path / 'train' / 'labels'
class_counts = Counter()

if train_labels_dir.exists():
    label_files = list(train_labels_dir.glob('*.txt'))
    for label_file in label_files:
        classes = parse_yolo_labels(label_file)
        for cls in classes:
            class_counts[cls] += 1

# İstatistikleri göster
total_samples = sum(class_counts.values())
num_classes = config['nc']
unique_classes = len(class_counts)

print(f"\n📊 Sınıf Dağılımı:")
print(f"   • Toplam annotation: {total_samples}")
print(f"   • Kullanılan sınıf: {unique_classes} / {num_classes}")
print(f"   • En yaygın sınıf: {max(class_counts.values())} örnek")
print(f"   • En nadir sınıf: {min(class_counts.values())} örnek")
print(f"   • Dengesizlik oranı: {max(class_counts.values()) / min(class_counts.values()):.1f}x")

# Class weights hesapla (inverse frequency)
# Formül: weight = total_samples / (num_classes * class_count)
class_weights = {}
for cls_id in range(num_classes):
    count = class_counts.get(cls_id, 1)  # Hiç örnek yoksa 1 varsay
    # Inverse frequency ile ağırlık hesapla
    weight = total_samples / (num_classes * count)
    class_weights[cls_id] = weight

# Ağırlıkları normalize et (ortalama=1)
weights_array = np.array([class_weights[i] for i in range(num_classes)])
weights_array = weights_array / weights_array.mean()

# Ağırlıkları sınırla (çok büyük değerler probleme sebep olabilir)
weights_array = np.clip(weights_array, 0.5, 10.0)

print(f"\n⚖️  Class Weights Hesaplandı:")
print(f"   • Min weight: {weights_array.min():.2f}")
print(f"   • Max weight: {weights_array.max():.2f}")
print(f"   • Mean weight: {weights_array.mean():.2f}")
print(f"   • Nadir sınıflar {weights_array.max():.1f}x daha fazla ağırlığa sahip")

# En yüksek ağırlıklı 10 sınıfı göster
top_weighted_classes = np.argsort(weights_array)[-10:][::-1]
print(f"\n🔝 En Yüksek Ağırlıklı 10 Sınıf:")
for idx, cls_id in enumerate(top_weighted_classes, 1):
    count = class_counts.get(cls_id, 0)
    weight = weights_array[cls_id]
    print(f"   {idx}. Sınıf {cls_id:3d}: {count:3d} örnek → Ağırlık: {weight:.2f}x")

print("\n✅ Class weights hazır - eğitimde kullanılacak")
print("   Not: YOLOv8, cls_gain parametresi ile otomatik ağırlıklandırma yapar")
print("   Ek olarak güçlü augmentation veri dengesizliğini azaltır")

In [ ]:
# 3. YOLOv8 Modelini Yükle
print("\n" + "="*60)
print("📦 MODEL YÜKLEME")
print("="*60)

# Pretrained YOLOv8n modelini yükle
model = YOLO('yolov8n.pt')
print("\n✅ YOLOv8n pretrained model yüklendi")
print("   Model: yolov8n.pt (nano - en hızlı)")

In [ ]:
# 4. Eğitim Parametrelerini Ayarla
print("\n" + "="*60)
print("⚙️  EĞİTİM PARAMETRELERİ")
print("="*60)

# Eğitim ayarları
EPOCHS = 100          # Epoch sayısı (100-300 arası önerilir)
BATCH_SIZE = 16       # Batch size (GPU RAM'e göre ayarlayın: 8, 16, 32)
IMG_SIZE = 640        # Görsel boyutu (640 veya 1280)
PATIENCE = 50         # Early stopping (50 epoch iyileşme olmazsa dur)
SAVE_DIR = 'runs/detect/train'  # Sonuçların kaydedileceği klasör

print(f"\n📋 Ayarlar:")
print(f"   • Epochs: {EPOCHS}")
print(f"   • Batch Size: {BATCH_SIZE}")
print(f"   • Image Size: {IMG_SIZE}")
print(f"   • Patience: {PATIENCE}")
print(f"   • Device: {'GPU' if device == 0 else 'CPU'}")
print(f"   • Mixed Precision (AMP): Aktif")
print(f"\n💾 Sonuçlar kaydedilecek: {SAVE_DIR}")

### 🎨 Veri Dengesizliği İçin Güçlendirilmiş Augmentation Stratejisi

In [ ]:
print("\n" + "="*60)
print("🎨 AUGMENTATION STRATEJİSİ (Veri Dengesizliği Çözümü)")
print("="*60)

# Veri dengesizliği için optimize edilmiş augmentation parametreleri
augmentation_strategy = {
    # ✅ RENK DÖNÜŞÜMLERI - Aydınlatma değişkenliğini artır
    'hsv_h': 0.025,        # Hue shift (0.015 → 0.025) - Renk tonunu değiştir
    'hsv_s': 0.8,          # Saturation (0.7 → 0.8) - Renk doygunluğu
    'hsv_v': 0.5,          # Value/Brightness (0.4 → 0.5) - Parlaklık
    
    # ✅ GEOMETRİK DÖNÜŞÜMLER - Pozisyon ve ölçek değişkenliği
    'degrees': 10.0,       # Rotation (0 → 10) - Hafif rotasyon ekle
    'translate': 0.15,     # Translation (0.1 → 0.15) - Daha fazla kaydırma
    'scale': 0.7,          # Scale (0.5 → 0.7) - Daha agresif ölçekleme
    'shear': 5.0,          # Shear (0 → 5) - Eğme/bükme ekle
    'perspective': 0.0005, # Perspective (0 → 0.0005) - Hafif perspektif
    
    # ✅ FLIP AUGMENTATIONS
    'flipud': 0.0,         # Vertical flip (kapalı - işaretler için uygun değil)
    'fliplr': 0.5,         # Horizontal flip (bazı işaretler için uygun)
    
    # ✅ ADVANCED AUGMENTATIONS - Nadir sınıflar için çok önemli!
    'mosaic': 1.0,         # Mosaic (aktif) - 4 görseli birleştirir
    'mixup': 0.15,         # MixUp (0 → 0.15) - Görselleri karıştır
    'copy_paste': 0.3,     # Copy-Paste (0 → 0.3) - Nesneleri kopyala-yapıştır
    
    # ✅ REGULARIZATION
    'erasing': 0.5,        # Random Erasing (0.4 → 0.5) - Rastgele bölge sil
    'auto_augment': 'randaugment',  # RandAugment kullan
    
    # ✅ DIĞER
    'close_mosaic': 15,    # Son 15 epoch mosaic kapat (10 → 15)
}

print("\n📊 Augmentation Detayları:")
print("\n1️⃣  RENK AUGMENTATIONS (Aydınlatma Değişkenliği):")
print(f"   • HSV-H (Renk Tonu): {augmentation_strategy['hsv_h']}")
print(f"   • HSV-S (Doygunluk): {augmentation_strategy['hsv_s']}")
print(f"   • HSV-V (Parlaklık): {augmentation_strategy['hsv_v']}")
print("   → Farklı ışık koşullarını simüle eder")

print("\n2️⃣  GEOMETRİK AUGMENTATIONS (Pozisyon/Ölçek):")
print(f"   • Rotation: ±{augmentation_strategy['degrees']}°")
print(f"   • Translation: {augmentation_strategy['translate']*100:.0f}%")
print(f"   • Scale: {augmentation_strategy['scale']}")
print(f"   • Shear: {augmentation_strategy['shear']}°")
print("   → Farklı açı ve uzaklıkları simüle eder")

print("\n3️⃣  ADVANCED AUGMENTATIONS (Veri Çoğaltma):")
print(f"   • Mosaic: {augmentation_strategy['mosaic']*100:.0f}% (4 görsel birleşir)")
print(f"   • MixUp: {augmentation_strategy['mixup']*100:.0f}% (görseller karışır)")
print(f"   • Copy-Paste: {augmentation_strategy['copy_paste']*100:.0f}% (nesneler kopyalanır)")
print("   → Nadir sınıflar için efektif örnek çoğaltma!")

print("\n4️⃣  REGULARIZATION:")
print(f"   • Random Erasing: {augmentation_strategy['erasing']*100:.0f}%")
print(f"   • Auto Augment: {augmentation_strategy['auto_augment']}")
print("   → Overfitting'i önler, generalization artırır")

print("\n✅ Augmentation stratejisi hazır!")
print("   💡 Bu ayarlar veri dengesizliğini önemli ölçüde azaltacak")
print("   💡 Nadir sınıflar için efektif eğitim örnekleri oluşturacak")

In [ ]:
# 5. EĞİTİMİ BAŞLAT
print("\n" + "="*60)
print("🚀 EĞİTİM BAŞLIYOR...")
print("="*60)
print("\n⏱️  Tahmini Süre: ~1-3 saat (GPU'ya göre değişir)")
print("   Not: İlk epoch yavaş başlar, sonra hızlanır\n")

# Modeli eğit
results = model.train(
    data=str(data_yaml),        # data.yaml dosyasının yolu
    epochs=EPOCHS,              # Epoch sayısı
    imgsz=IMG_SIZE,             # Görsel boyutu
    batch=BATCH_SIZE,           # Batch size
    device=device,              # GPU veya CPU
    patience=PATIENCE,          # Early stopping
    save=True,                  # Model kaydet
    project='runs/detect',      # Proje klasörü
    name='train',               # Eğitim ismi
    exist_ok=True,              # Mevcut klasörün üzerine yaz
    pretrained=True,            # Pretrained weights kullan
    optimizer='auto',           # Otomatik optimizer seçimi
    verbose=True,               # Detaylı log
    seed=42,                    # Reproducibility
    deterministic=True,         # Sabit sonuçlar
    single_cls=False,           # Multi-class
    rect=False,                 # Rectangular training (kapalı)
    cos_lr=False,               # Cosine learning rate (kapalı)
    close_mosaic=15,            # Son 15 epoch mosaic kapalı (10 → 15) ⚖️ DEĞİŞTİRİLDİ
    resume=False,               # Yeni eğitim
    amp=True,                   # Mixed precision (hızlandırma)
    fraction=1.0,               # Veri setinin %100'ünü kullan
    profile=False,              # Profiling (kapalı)
    freeze=None,                # Freeze layers (yok)
    lr0=0.01,                   # Initial learning rate
    lrf=0.01,                   # Final learning rate
    momentum=0.937,             # SGD momentum
    weight_decay=0.0005,        # Weight decay
    warmup_epochs=3.0,          # Warmup epochs
    warmup_momentum=0.8,        # Warmup momentum
    warmup_bias_lr=0.1,         # Warmup bias learning rate
    box=7.5,                    # Box loss gain
    cls=1.0,                    # Class loss gain (0.5 → 1.0) ⚖️ VERİ DENGESİZLİĞİ İÇİN ARTIRILDI
    dfl=1.5,                    # DFL loss gain
    pose=12.0,                  # Pose loss gain (kullanılmıyor)
    kobj=1.0,                   # Keypoint obj loss gain (kullanılmıyor)
    label_smoothing=0.0,        # Label smoothing epsilon
    nbs=64,                     # Nominal batch size
    
    # 🎨 GÜÇLENDİRİLMİŞ AUGMENTATION (Veri Dengesizliği Çözümü)
    hsv_h=0.025,                # HSV-Hue (0.015 → 0.025) - Daha fazla renk değişimi
    hsv_s=0.8,                  # HSV-Saturation (0.7 → 0.8)
    hsv_v=0.5,                  # HSV-Value (0.4 → 0.5)
    degrees=10.0,               # Rotation (0 → 10) - Rotasyon eklendi
    translate=0.15,             # Translation (0.1 → 0.15) - Daha fazla kaydırma
    scale=0.7,                  # Scale (0.5 → 0.7) - Daha agresif ölçekleme
    shear=5.0,                  # Shear (0 → 5) - Eğme eklendi
    perspective=0.0005,         # Perspective (0 → 0.0005) - Perspektif eklendi
    flipud=0.0,                 # Vertical flip (kapalı)
    fliplr=0.5,                 # Horizontal flip
    mosaic=1.0,                 # Mosaic augmentation
    mixup=0.15,                 # MixUp (0 → 0.15) ⚖️ NADİR SINIFLAR İÇİN EKLENDİ
    copy_paste=0.3,             # Copy-paste (0 → 0.3) ⚖️ NADİR SINIFLAR İÇİN EKLENDİ
    auto_augment='randaugment', # Auto augmentation
    erasing=0.5,                # Random erasing (0.4 → 0.5)
    crop_fraction=1.0,          # Crop fraction
    plots=True,                 # Grafikleri kaydet
    val=True                    # Validation yap
)

print("\n" + "="*60)
print("✅ EĞİTİM TAMAMLANDI!")
print("="*60)

---
### ✅ Veri Dengesizliği Çözümü Uygulandı!

**Yapılan İyileştirmeler:**
1. **Class Loss Gain:** `0.5 → 1.0` (2x artış) - Nadir sınıfların daha iyi öğrenilmesini sağlar
2. **MixUp Augmentation:** `0% → 15%` - Görselleri karıştırarak yeni örnekler oluşturur
3. **Copy-Paste:** `0% → 30%` - Nesneleri kopyalayıp yapıştırarak örnekleri çoğaltır
4. **Güçlendirilmiş Geometrik Augmentations:** Rotasyon, shear, perspektif eklendi
5. **Renk Augmentations:** Daha agresif HSV değişimleri

**Beklenen Faydalar:**
- 🎯 Nadir sınıfların performansı artacak (1-10 örnek olan sınıflar)
- 📈 Genel mAP'te %5-10 iyileşme bekleniyor
- ⚖️ Confusion matrix'te daha dengeli sonuçlar
- 🚀 Gerçek dünya senaryolarında daha robust model

---

In [ ]:
# 6. Eğitim Sonuçlarını Görselleştir
from PIL import Image
import matplotlib.pyplot as plt

print("\n📊 EĞİTİM SONUÇLARI\n")

# Results.png dosyasını göster
results_img_path = Path('runs/detect/train/results.png')
if results_img_path.exists():
    img = Image.open(results_img_path)
    plt.figure(figsize=(20, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("✅ Eğitim grafikleri gösterildi")
else:
    print("⚠️  results.png bulunamadı")

# Confusion matrix
confusion_matrix_path = Path('runs/detect/train/confusion_matrix.png')
if confusion_matrix_path.exists():
    img = Image.open(confusion_matrix_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("✅ Confusion matrix gösterildi")

In [ ]:
# 7. En İyi Modeli Yükle ve Validasyon Yap
print("\n" + "="*60)
print("🎯 VALİDASYON")
print("="*60)

# En iyi modeli yükle
best_model_path = Path('runs/detect/train/weights/best.pt')
if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print(f"\n✅ En iyi model yüklendi: {best_model_path}")
    
    # Validation set üzerinde test et
    print("\n⏳ Validation yapılıyor...")
    metrics = best_model.val(data=str(data_yaml))
    
    # Metrikleri göster
    print("\n📈 PERFORMANS METRİKLERİ:")
    print("="*60)
    print(f"   mAP50-95: {metrics.box.map:.4f}    (Ana metrik)")
    print(f"   mAP50:    {metrics.box.map50:.4f}  (IoU=0.5)")
    print(f"   mAP75:    {metrics.box.map75:.4f}  (IoU=0.75)")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall:    {metrics.box.mr:.4f}")
    print("="*60)
else:
    print(f"❌ Model bulunamadı: {best_model_path}")

In [ ]:
# 8. Test Set Üzerinde Tahmin Yap
import random
import cv2

print("\n" + "="*60)
print("🔮 TEST TAHMİNLERİ")
print("="*60)

test_images_path = dataset_path / 'test' / 'images'
test_images = list(test_images_path.glob('*.jpg')) + list(test_images_path.glob('*.jpeg')) + list(test_images_path.glob('*.png'))

if len(test_images) > 0 and best_model_path.exists():
    # Rastgele 6 görsel seç
    sample_images = random.sample(test_images, min(6, len(test_images)))
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    print(f"\n⏳ {len(sample_images)} test görseli üzerinde tahmin yapılıyor...\n")
    
    for idx, img_path in enumerate(sample_images):
        # Tahmin yap
        results = best_model.predict(img_path, conf=0.25, verbose=False)
        
        # Sonucu görselleştir
        result_img = results[0].plot()
        result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(result_img_rgb)
        axes[idx].set_title(f'{img_path.name}', fontsize=10)
        axes[idx].axis('off')
    
    # Boş subplot'ları gizle
    for idx in range(len(sample_images), 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    print("✅ Test tahminleri gösterildi")
else:
    print("⚠️  Test görselleri bulunamadı veya model yok")

In [ ]:
# 9. Modeli Farklı Formatlarda Dışa Aktar
print("\n" + "="*60)
print("💾 MODEL EXPORT")
print("="*60)

if best_model_path.exists():
    print("\n⏳ Model farklı formatlara dönüştürülüyor...\n")
    
    # ONNX formatına çevir (evrensel format)
    try:
        onnx_path = best_model.export(format='onnx')
        print(f"✅ ONNX: {onnx_path}")
    except Exception as e:
        print(f"❌ ONNX export hatası: {e}")
    
    # TorchScript formatına çevir (PyTorch deployment)
    try:
        torchscript_path = best_model.export(format='torchscript')
        print(f"✅ TorchScript: {torchscript_path}")
    except Exception as e:
        print(f"❌ TorchScript export hatası: {e}")
    
    print("\n" + "="*60)
    print("✅ EXPORT TAMAMLANDI")
    print("="*60)

In [ ]:
# 10. Final Özet
print("\n" + "="*80)
print("🎉 MODEL EĞİTİMİ BAŞARIYLA TAMAMLANDI!")
print("="*80)

print(f"\n📂 Dosya Konumları:")
print(f"   • En İyi Model: runs/detect/train/weights/best.pt")
print(f"   • Son Model: runs/detect/train/weights/last.pt")
print(f"   • Sonuçlar: runs/detect/train/results.png")
print(f"   • Grafikler: runs/detect/train/*.png")

if best_model_path.exists():
    print(f"\n📊 Model Performansı:")
    print(f"   • mAP50-95: {metrics.box.map:.4f}")
    print(f"   • mAP50: {metrics.box.map50:.4f}")
    print(f"   • Precision: {metrics.box.mp:.4f}")
    print(f"   • Recall: {metrics.box.mr:.4f}")

print(f"\n🚀 Modeli Kullanma:")
print(f"   from ultralytics import YOLO")
print(f"   model = YOLO('runs/detect/train/weights/best.pt')")
print(f"   results = model.predict('image.jpg')")

print(f"\n💡 Öneriler:")
print(f"   • Model yeterince iyi değilse epochs sayısını artırın (200-300)")
print(f"   • Overfitting varsa data augmentation parametrelerini artırın")
print(f"   • Underfitting varsa daha büyük model kullanın (yolov8s.pt, yolov8m.pt)")
print(f"   • GPU RAM yetersizse batch size'ı azaltın (8 veya 4)")

print("\n" + "="*80)